# EDA - Teen Mental Health Dataset

Este notebook presenta un analisis exploratorio del dataset `Teen_Mental_Health_Dataset.csv`. El objetivo es estudiar la relacion entre habitos digitales, sueno, actividad fisica, rendimiento academico y la variable objetivo `depression_label`.

El analisis parte del borrador inicial y lo reorganiza en un flujo reproducible: carga de datos, validacion, limpieza, feature engineering, visualizacion y contraste estadistico basico.


## 1. Importacion de librerias y configuracion


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.cleaning import clean_data, detect_outliers_iqr
from src.features import build_features
from src.io import load_data, save_data
from src.utils import chi_square_report, missing_values_report
from src.viz import (
    plot_correlation_matrix,
    plot_depression_rate_by_category,
    plot_depression_vs_social_media,
    plot_numeric_distributions,
)

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "Teen_Mental_Health_Dataset.csv"
PROCESSED_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "teen_mental_health_processed.csv"

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", None)


## 2. Carga de datos

Se carga el CSV original desde `data/raw`. Esta version se mantiene sin modificar para conservar una fuente de datos reproducible.


In [ ]:
df_raw = load_data(RAW_DATA_PATH)
df_raw.head()


In [ ]:
print(f"Filas: {df_raw.shape[0]}")
print(f"Columnas: {df_raw.shape[1]}")
df_raw.info()


## 3. Validacion inicial

Antes de transformar el dataset se revisan columnas, nulos y duplicados. En esta muestra no aparecen valores nulos ni registros duplicados, pero se mantiene el paso de limpieza dentro del pipeline para que el proyecto sea robusto.


In [ ]:
df_raw.columns.tolist()


In [ ]:
missing_values_report(df_raw)


In [ ]:
print(f"Duplicados: {df_raw.duplicated().sum()}")


## 4. Limpieza y feature engineering

Se normalizan nombres de columnas, se convierten variables categoricas y se crean variables derivadas:

- `heavy_social_media_user`: marca usuarios con 5 o mas horas diarias en redes sociales.
- `sleep_quality`: clasifica el sueno en `poor`, `normal` y `good`.
- `risk_score`: indicador sintetico basado en redes sociales, sueno y rendimiento academico.
- `risk_category`: terciles de `risk_score`.


In [ ]:
df = clean_data(df_raw)
df = build_features(df)
save_data(df, PROCESSED_DATA_PATH)

df.head()


In [ ]:
print(f"Dataset procesado guardado en: {PROCESSED_DATA_PATH}")
print(f"Filas: {df.shape[0]} | Columnas: {df.shape[1]}")
df.dtypes


## 5. Analisis descriptivo

Se revisan estadisticos generales de las variables numericas y la distribucion de la variable objetivo. El porcentaje de positivos en `depression_label` es bajo, por lo que cualquier comparacion debe interpretarse teniendo en cuenta el desbalance.


In [ ]:
df.describe().T


In [ ]:
target_distribution = (
    df["depression_label"]
    .value_counts(normalize=True)
    .mul(100)
    .rename("percentage")
    .to_frame()
)
target_distribution


In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="depression_label")
plt.title("Distribucion de depression_label")
plt.xlabel("Depression label")
plt.ylabel("Frecuencia")
plt.tight_layout()
plt.show()


## 6. Distribuciones numericas

Los histogramas permiten detectar rangos, concentraciones y posibles variables con distribuciones poco uniformes.


In [ ]:
numeric_cols = df.select_dtypes(include="number").columns.tolist()
plot_numeric_distributions(df, numeric_cols)


## 7. Variables categoricas

Se revisa la frecuencia de genero, plataforma principal y nivel de interaccion social.


In [ ]:
categorical_cols = ["gender", "platform_usage", "social_interaction_level"]

fig, axes = plt.subplots(1, len(categorical_cols), figsize=(16, 4))

for ax, col in zip(axes, categorical_cols):
    order = df[col].value_counts().index
    sns.countplot(data=df, x=col, order=order, ax=ax)
    ax.set_title(col)
    ax.tick_params(axis="x", rotation=25)

plt.tight_layout()
plt.show()


## 8. Correlaciones

La matriz de correlacion ayuda a observar asociaciones lineales entre variables numericas. No implica causalidad, pero orienta el analisis posterior.


In [ ]:
plot_correlation_matrix(df, numeric_cols)


In [ ]:
(
    df[numeric_cols]
    .corr()["depression_label"]
    .sort_values(ascending=False)
    .to_frame("correlation_with_depression_label")
)


## 9. Relacion entre variables numericas y depression_label

Se comparan algunas variables clave frente a la etiqueta de depresion: uso diario de redes sociales, horas de sueno, estres y ansiedad.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
variables_to_compare = [
    "daily_social_media_hours",
    "sleep_hours",
    "stress_level",
    "anxiety_level",
]

for ax, col in zip(axes.ravel(), variables_to_compare):
    sns.boxplot(data=df, x="depression_label", y=col, ax=ax)
    ax.set_title(f"{col} vs depression_label")

plt.tight_layout()
plt.show()


In [ ]:
plot_depression_vs_social_media(df)


## 10. Relacion entre variables categoricas y depression_label

Se calcula la tasa media de `depression_label` por categoria y se contrastan asociaciones con chi-cuadrado y Cramer's V.


In [ ]:
for col in categorical_cols:
    display(
        df.groupby(col, observed=True)["depression_label"]
        .agg(["mean", "count"])
        .sort_values("mean", ascending=False)
    )


In [ ]:
plot_depression_rate_by_category(df, categorical_cols)


In [ ]:
chi_square_report(df, categorical_cols, "depression_label")


## 11. Outliers

Se revisan outliers con el metodo IQR en variables numericas principales. En este contexto no se eliminan automaticamente: primero conviene decidir si representan errores o casos reales.


In [ ]:
outlier_summary = []

for col in numeric_cols:
    outliers = detect_outliers_iqr(df, col)
    outlier_summary.append({"feature": col, "outliers": len(outliers)})

pd.DataFrame(outlier_summary).sort_values("outliers", ascending=False)


## 12. Conclusiones

- El dataset tiene 1200 registros y 13 variables originales; tras feature engineering se incorporan variables derivadas para facilitar el analisis.
- No se observan nulos ni duplicados en la version revisada.
- `depression_label` esta desbalanceada: los positivos representan aproximadamente un 2.6% de la muestra.
- Las asociaciones lineales mas visibles con `depression_label` son: mayor uso diario de redes sociales, mayor estres, mayor ansiedad y menos horas de sueno.
- Las variables categoricas revisadas no muestran una asociacion fuerte con `depression_label` segun chi-cuadrado y Cramer's V.
- Los resultados son exploratorios y no deben interpretarse como causalidad.
